# Préparation des données d'usage

Données fictives. Le notebook lit les fichiers bruts du Lakehouse, corrige les anomalies, contrôle la qualité, puis écrit les tables Delta du modèle en étoile.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

DOSSIER = "Files/raw"
FORMAT_TABLE = "delta"
# Hypothèse calée sur les traces de l'agent : environ 0,001 $ par question.
PRIX_ENTREE_1M = 0.25
PRIX_SORTIE_1M = 2.00
STATUTS = ["reponse", "refus", "escalade"]

def lire_csv(nom, schema=None):
    lecteur = spark.read.option("header", True).option("encoding", "UTF-8")
    lecteur = lecteur.schema(schema) if schema else lecteur.option("inferSchema", False)
    return lecteur.csv(f"{DOSSIER}/{nom}")

def ecrire_table(df, nom):
    df.write.format(FORMAT_TABLE).mode("overwrite").option("overwriteSchema", "true").saveAsTable(nom)
    print(f"{nom} : {df.count()} lignes")

## Lecture

In [ ]:
schema_questions = StructType([
    StructField("question_id", StringType()),
    StructField("horodatage", StringType()),
    StructField("conseiller_id", StringType()),
    StructField("theme", StringType()),
    StructField("loi", StringType()),
    StructField("langue", StringType()),
    StructField("statut", StringType()),
    StructField("latence_ms", IntegerType()),
    StructField("tokens_entree", IntegerType()),
    StructField("tokens_sortie", IntegerType()),
    StructField("feedback", StringType()),
])

brut = lire_csv("questions.csv", schema_questions)
agences = lire_csv("agences.csv")
conseillers = lire_csv("conseillers.csv")
habilitations = lire_csv("habilitations.csv")
print("Lignes brutes :", brut.count())

## Nettoyage

In [ ]:
normalise = (
    brut
    .dropDuplicates(["question_id"])
    .withColumn("statut", F.lower(F.trim("statut")))
    .withColumn("horodatage", F.to_timestamp("horodatage", "yyyy-MM-dd HH:mm:ss"))
)

connus = conseillers.select("conseiller_id").withColumn("conseiller_connu", F.lit(True))
controle = (
    normalise.join(connus, "conseiller_id", "left")
    .withColumn(
        "motif_rejet",
        F.when(F.col("horodatage").isNull(), "horodatage_invalide")
        .when(F.col("conseiller_connu").isNull(), "conseiller_inconnu")
        .when(F.col("latence_ms") <= 0, "latence_invalide")
        .when(~F.col("statut").isin(STATUTS), "statut_inconnu"),
    )
    .drop("conseiller_connu")
)

rejets = controle.filter(F.col("motif_rejet").isNotNull())
valides = controle.filter(F.col("motif_rejet").isNull()).drop("motif_rejet")
print("Doublons retirés :", brut.count() - normalise.count())
rejets.groupBy("motif_rejet").count().show()

## Modèle en étoile

In [ ]:
fact_questions = (
    valides
    .withColumn("date", F.to_date("horodatage"))
    .withColumn("heure", F.hour("horodatage"))
    .withColumn("feedback", F.when(F.col("feedback").isin("utile", "pas_utile"), F.col("feedback")))
    .withColumn(
        "cout_estime_usd",
        F.round(
            (F.col("tokens_entree") * PRIX_ENTREE_1M + F.col("tokens_sortie") * PRIX_SORTIE_1M) / 1_000_000,
            6,
        ),
    )
    .join(conseillers.select("conseiller_id", "agence_id"), "conseiller_id")
)

bornes = fact_questions.agg(F.min("date").alias("debut"), F.max("date").alias("fin")).first()
dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{bornes.debut}'), to_date('{bornes.fin}'), interval 1 day)) AS date")
    .withColumn("annee", F.year("date"))
    .withColumn("mois", F.month("date"))
    .withColumn("nom_mois", F.date_format("date", "yyyy-MM"))
    .withColumn("semaine", F.weekofyear("date"))
)
dim_agence = agences
dim_conseiller = conseillers

## Contrôles de qualité

L'écriture s'arrête si un contrôle échoue.

In [ ]:
controles = {
    "identifiants uniques": fact_questions.count() == fact_questions.select("question_id").distinct().count(),
    "agence renseignée": fact_questions.filter(F.col("agence_id").isNull()).count() == 0,
    "statuts valides": fact_questions.filter(~F.col("statut").isin(STATUTS)).count() == 0,
    "latences positives": fact_questions.filter(F.col("latence_ms") <= 0).count() == 0,
    "dates couvertes": fact_questions.join(dim_date, "date", "left_anti").count() == 0,
}
for nom, reussi in controles.items():
    print(("OK " if reussi else "ECHEC"), nom)
if not all(controles.values()):
    raise ValueError("Contrôle de qualité en échec : aucune table n'est écrite.")

## Écriture des tables

In [ ]:
ecrire_table(dim_agence, "dim_agence")
ecrire_table(dim_conseiller, "dim_conseiller")
ecrire_table(dim_date, "dim_date")
ecrire_table(fact_questions, "fact_questions")
ecrire_table(habilitations, "habilitations")
ecrire_table(rejets, "rejets_questions")